# CardioIA Fase 2 - Ir Além 2: Rede Neural MLP para Classificação de ECG

## Diagnóstico Visual em Cardiologia

Este notebook implementa uma Rede Neural Artificial (MLP) para classificar sinais de ECG como **normal** ou **anormal**.

**Dataset:** MIT-BIH Arrhythmia Database (via Kaggle Heartbeat Dataset)  
- Fonte: https://www.kaggle.com/datasets/shayanfazeli/heartbeat  
- Licença: Open Database License (ODbL)  
- Referência: Kachuee et al., 2018 — ECG Heartbeat Classification

**Arquitetura:** MLP (Multi-Layer Perceptron) com Scikit-learn  
> **Nota:** Implementado com `sklearn.neural_network.MLPClassifier` para compatibilidade máxima.  
> Para uso com TensorFlow/Keras, recomenda-se Python 3.10–3.12.

**Autor:** CardioIA Team | **Versão:** 1.0

---
## 1. Imports e Configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✓ Imports concluídos!')
print('  Usando: sklearn.neural_network.MLPClassifier')

---
## 2. Carregamento do Dataset

O dataset **Heartbeat** do Kaggle contém sinais de ECG pré-processados do MIT-BIH Arrhythmia Database.

Cada linha representa um batimento cardíaco com **187 features** (amostras do sinal) e uma **classe**:
- **0**: Normal
- **1–4**: Diferentes tipos de arritmia

Para este projeto, realizamos **classificação binária**: Normal (0) vs. Anormal (1–4).

> Se o dataset não estiver disponível localmente, geramos dados sintéticos que simulam as características do ECG.

In [ ]:
CAMINHO_TRAIN = '../data/mitbih_train.csv'
CAMINHO_TEST  = '../data/mitbih_test.csv'
N_FEATURES    = 187


def gerar_ecg_sintetico(n_normal=4000, n_anormal=4000, n_features=N_FEATURES):
    """
    Gera sinais de ECG sintéticos para demonstração.
    Normal: sinal periódico com pico QRS bem definido.
    Anormal: sinal com irregularidades de amplitude, posição ou forma.
    """
    print('⚠️  Dataset MIT-BIH não encontrado. Gerando dados sintéticos...')
    print('   Para dados reais: https://www.kaggle.com/datasets/shayanfazeli/heartbeat')
    print('   Coloque mitbih_train.csv e mitbih_test.csv em data/')
    print()

    t = np.linspace(0, 1, n_features)

    normais = []
    for _ in range(n_normal):
        p    = 0.15 * np.exp(-((t - 0.20) ** 2) / 0.002)
        qrs  = 1.00 * np.exp(-((t - 0.50) ** 2) / 0.0005)
        tw   = 0.30 * np.exp(-((t - 0.70) ** 2) / 0.003)
        noise = np.random.normal(0, 0.02, n_features)
        normais.append(p + qrs + tw + noise)

    anormais = []
    for i in range(n_anormal):
        tipo = i % 4
        if tipo == 0:   # Amplitude reduzida
            amp = np.random.uniform(0.3, 0.6)
            qrs = amp * np.exp(-((t - 0.50) ** 2) / 0.0005)
        elif tipo == 1: # Pico deslocado
            s = np.random.uniform(-0.15, 0.15)
            qrs = 1.0 * np.exp(-((t - (0.5 + s)) ** 2) / 0.0005)
        elif tipo == 2: # Pico alargado
            qrs = 0.8 * np.exp(-((t - 0.50) ** 2) / 0.003)
        else:           # Duplo pico
            qrs = (0.7 * np.exp(-((t - 0.40) ** 2) / 0.0005) +
                   0.5 * np.exp(-((t - 0.60) ** 2) / 0.0005))
        noise = np.random.normal(0, 0.05, n_features)
        anormais.append(qrs + noise)

    X = np.vstack([normais, anormais])
    y = np.array([0] * n_normal + [1] * n_anormal)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]


def carregar_dataset():
    if os.path.exists(CAMINHO_TRAIN) and os.path.exists(CAMINHO_TEST):
        print('✓ Dataset MIT-BIH encontrado. Carregando...')
        df_tr = pd.read_csv(CAMINHO_TRAIN, header=None)
        df_te = pd.read_csv(CAMINHO_TEST,  header=None)
        X_tr, y_tr = df_tr.iloc[:, :-1].values, (df_tr.iloc[:, -1].values > 0).astype(int)
        X_te, y_te = df_te.iloc[:, :-1].values, (df_te.iloc[:, -1].values > 0).astype(int)
        print(f'  Treino: {X_tr.shape} | Teste: {X_te.shape}')
        return X_tr, X_te, y_tr, y_te, True
    else:
        X, y = gerar_ecg_sintetico()
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )
        return X_tr, X_te, y_tr, y_te, False


X_train, X_test, y_train, y_test, dados_reais = carregar_dataset()
print(f'Treino : {X_train.shape} | Normal={np.sum(y_train==0)}, Anormal={np.sum(y_train==1)}')
print(f'Teste  : {X_test.shape}  | Normal={np.sum(y_test==0)},  Anormal={np.sum(y_test==1)}')

---
## 3. Visualização dos Sinais de ECG

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
fig.suptitle('Exemplos de Sinais de ECG', fontsize=14, fontweight='bold')

for col, (classe, label, cor) in enumerate([(0, 'Normal', '#2ecc71'), (1, 'Anormal', '#e74c3c')]):
    idx_classe = np.where(y_train == classe)[0][:2]
    for row, idx in enumerate(idx_classe):
        axes[row][col].plot(X_train[idx], color=cor, linewidth=1.2)
        axes[row][col].set_title(f'{label} — Exemplo {row+1}', color=cor, fontweight='bold')
        axes[row][col].set_xlabel('Amostras')
        axes[row][col].set_ylabel('Amplitude')
        axes[row][col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/exemplos_ecg.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Gráfico salvo em data/exemplos_ecg.png')

---
## 4. Pré-processamento

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Shape treino normalizado: {X_train_scaled.shape}')
print(f'Média (deve ser ~0)     : {X_train_scaled.mean():.6f}')
print(f'Desvio padrão (~1)      : {X_train_scaled.std():.6f}')

---
## 5. Arquitetura da Rede Neural MLP

Implementamos um **Perceptron Multicamadas (MLP)** com:
- Camadas ocultas: `(256, 128, 64, 32)` neurônios
- Ativação: **ReLU**
- Regularização: **L2** (alpha)
- Otimizador: **Adam**
- Saída: **Sigmoid** (classificação binária)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,          # Regularização L2
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=False
)

print('Arquitetura MLP:')
print(f'  Camadas ocultas : {mlp.hidden_layer_sizes}')
print(f'  Ativação        : {mlp.activation}')
print(f'  Otimizador      : {mlp.solver}')
print(f'  Regularização   : L2 (alpha={mlp.alpha})')
print(f'  Early Stopping  : {mlp.early_stopping}')

---
## 6. Treinamento

In [ ]:
print('Treinando MLP...')
mlp.fit(X_train_scaled, y_train)
print(f'✓ Treinamento concluído em {mlp.n_iter_} iterações!')

---
## 7. Curva de Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mlp.loss_curve_, color='#3498db', linewidth=2, label='Loss de Treino')
if mlp.validation_scores_ is not None:
    ax2 = ax.twinx()
    ax2.plot(mlp.validation_scores_, color='#e74c3c', linewidth=2, linestyle='--', label='Acurácia Validação')
    ax2.set_ylabel('Acurácia de Validação', color='#e74c3c')
    ax2.legend(loc='lower right')
ax.set_title('Curva de Loss durante Treinamento', fontweight='bold')
ax.set_xlabel('Iteração')
ax.set_ylabel('Loss', color='#3498db')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/curvas_aprendizado.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Gráfico salvo em data/curvas_aprendizado.png')

---
## 8. Avaliação no Conjunto de Teste

In [ ]:
y_pred = mlp.predict(X_test_scaled)
y_prob = mlp.predict_proba(X_test_scaled)[:, 1]

from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print('=' * 60)
print('RESULTADOS NO CONJUNTO DE TESTE')
print('=' * 60)
print(f'  Acurácia : {acc:.4f} ({acc*100:.2f}%)')
print(f'  AUC-ROC  : {auc:.4f}')
print()
print('Relatório de Classificação:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anormal']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anormal']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Matriz de Confusão', fontweight='bold')

# Distribuição de probabilidades
axes[1].hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='#2ecc71', label='Normal',  edgecolor='black')
axes[1].hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='#e74c3c', label='Anormal', edgecolor='black')
axes[1].axvline(0.5, color='black', linestyle='--', label='Limiar (0.5)')
axes[1].set_title('Distribuição de Probabilidades', fontweight='bold')
axes[1].set_xlabel('P(Anormal)')
axes[1].set_ylabel('Frequência')
axes[1].legend()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[2].plot(fpr, tpr, color='#3498db', linewidth=2, label=f'AUC = {auc:.3f}')
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatório')
axes[2].set_title('Curva ROC', fontweight='bold')
axes[2].set_xlabel('Taxa de Falsos Positivos')
axes[2].set_ylabel('Taxa de Verdadeiros Positivos')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/avaliacao_mlp.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Gráfico salvo em data/avaliacao_mlp.png')

---
## 9. Classificação de Novos Sinais

In [ ]:
def classificar_ecg(sinal, modelo, scaler):
    """
    Classifica um sinal de ECG como Normal ou Anormal.
    """
    sinal_norm = scaler.transform(sinal.reshape(1, -1))
    prob  = modelo.predict_proba(sinal_norm)[0][1]
    classe = 'Anormal' if prob >= 0.5 else 'Normal'
    return {'classe': classe, 'probabilidade': float(prob), 'confianca': float(max(prob, 1 - prob))}


print('Classificação de exemplos do conjunto de teste:')
print('=' * 65)
for i in range(8):
    r    = classificar_ecg(X_test[i], mlp, scaler)
    real = 'Normal' if y_test[i] == 0 else 'Anormal'
    ok   = '✓' if r['classe'] == real else '✗'
    print(f'  Exemplo {i+1}: Predito={r["classe"]:8s} | Real={real:8s} | '
          f'Confiança={r["confianca"]*100:.1f}% {ok}')

---
## 10. Conclusão

### Resultados

A rede neural MLP demonstrou capacidade de classificar sinais de ECG:
- **Arquitetura**: 4 camadas ocultas (256→128→64→32) com ReLU
- **Regularização**: L2 (alpha) e Early Stopping
- **Classificação binária**: Normal vs. Anormal

### Dataset
- **MIT-BIH Arrhythmia Database** (via Kaggle Heartbeat)
- Referência: Kachuee, M., et al. (2018). ECG Heartbeat Classification: A Deep Transferable Representation.
- URL: https://www.kaggle.com/datasets/shayanfazeli/heartbeat

### Nota sobre Implementação
Este notebook usa `sklearn.neural_network.MLPClassifier` por compatibilidade com Python 3.14.  
Para TensorFlow/Keras, recomenda-se Python 3.10–3.12 com `pip install tensorflow`.

### Considerações Éticas
- Este modelo é uma **demonstração acadêmica**
- Uso clínico real requer validação rigorosa e aprovação regulatória (ANVISA)
- Dados de saúde estão sujeitos à **LGPD**
- Falsos negativos têm consequências graves em contexto clínico

### Limitações
- MLP não captura dependências temporais (CNN/LSTM seriam mais adequados)
- Dataset pode ter viés de seleção
- Generalização para outros equipamentos não garantida